# Quran Forced Alignment — All Reciters × 114 Surahs (Colab GPU)

High-throughput, overlapped asynchronous forced alignment and audio normalization pipeline.

### Pipeline Architecture:
- **Stage 1 (Async Prefetch):** High-speed async audio downloader / cache prefetch via `curl_cffi`.
- **Stage 2 (Audio Transcode):** Multithreaded ffmpeg decode + EBU R128 loudnorm & Opus transcoding.
- **Stage 3 (Feature Extraction):** Deterministic 80-channel Fbank feature extraction + whole-surah phoneme reference + silence split points.
- **Stage 4 (Batched GPU Inference):** ONNX Runtime CUDA streaming Zipformer2-CTC (`Quran-Lab/zipformer_p-arabic-v3.int8.onnx`) + `torchaudio.functional.forced_align`.
- **Stage 5 (Export & Sync):** Timed word-level JSON + Tajweed/letter tier + SRT + Opus → Google Drive.

**GPU Config:** `--device cuda --cuda-batch-size 8 --intra-surah-split --prefetch-workers 4 --prefetch-batches 2`

In [ ]:
# 1. Install package + CUDA extra & dependencies
!pip install -q git+https://github.com/HsnSaboor/quran-forced-align.git --extra cuda 2>/dev/null || \
!pip install -q -e . --extra cuda
!pip install -q curl_cffi tqdm huggingface_hub
!apt-get install -y -qq ffmpeg > /dev/null 2>&1
print("✓ Environment and CUDA dependencies installed.")

In [ ]:
# 2. Mount Google Drive & configure storage layout
import os
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT   = '/content/drive/MyDrive/quran_forced_align'
DRIVE_MODEL  = f'{DRIVE_ROOT}/model'
DRIVE_OPUS   = f'{DRIVE_ROOT}/opus'
DRIVE_JSON   = f'{DRIVE_ROOT}/json'
DRIVE_AUDIO  = f'{DRIVE_ROOT}/audio_input'

# Local high-speed NVMe scratch cache
LOCAL_AUDIO  = '/content/audio_cache'
LOCAL_OUTPUT = '/content/output_cache'
LOCAL_MODEL  = '/content/model'

for d in [DRIVE_ROOT, DRIVE_MODEL, DRIVE_OPUS, DRIVE_JSON, DRIVE_AUDIO, LOCAL_AUDIO, LOCAL_OUTPUT, LOCAL_MODEL]:
    os.makedirs(d, exist_ok=True)

print(f"✓ Drive mounted. Output root: {DRIVE_ROOT}")

In [ ]:
# 3. Automated Model Download: Quran-Lab/zipformer_p-arabic-v3 + Drive Cache
import os, shutil
from huggingface_hub import hf_hub_download

MODEL_REPO = "Quran-Lab/zipformer_p-arabic-v3"
MODEL_FILE = "zipformer_p_arabic_v3.int8.onnx"
TOKENS_FILE = "tokens.txt"

local_model_path = os.path.join(LOCAL_MODEL, MODEL_FILE)
local_tokens_path = os.path.join(LOCAL_MODEL, TOKENS_FILE)
drive_model_path = os.path.join(DRIVE_MODEL, MODEL_FILE)
drive_tokens_path = os.path.join(DRIVE_MODEL, TOKENS_FILE)

# Check Colab Secrets or Env for HF token
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN')

# 1. Try restoring from Google Drive cache
if os.path.exists(drive_model_path) and os.path.exists(drive_tokens_path):
    print(f"✓ Found model in Google Drive cache: {drive_model_path}")
    shutil.copy2(drive_model_path, local_model_path)
    shutil.copy2(drive_tokens_path, local_tokens_path)
    print("✓ Copied model to local NVMe storage.")
else:
    print(f"Downloading {MODEL_FILE} from Hugging Face ({MODEL_REPO})...")
    dl_model = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE, token=hf_token)
    dl_tokens = hf_hub_download(repo_id=MODEL_REPO, filename=TOKENS_FILE, token=hf_token)
    
    shutil.copy2(dl_model, local_model_path)
    shutil.copy2(dl_tokens, local_tokens_path)
    
    # Save to Drive cache for future sessions
    shutil.copy2(dl_model, drive_model_path)
    shutil.copy2(dl_tokens, drive_tokens_path)
    print(f"✓ Model downloaded and cached to Google Drive: {DRIVE_MODEL}")

print(f"✓ Model ready: {local_model_path} ({os.path.getsize(local_model_path)/1024/1024:.1f} MB)")

In [ ]:
# 4. Reciter Configuration & High-Speed Async Downloader (curl_cffi)
import asyncio, json, glob
from curl_cffi.requests import AsyncSession
from tqdm.auto import tqdm

RECITERS = [
    {"name": "mishary-rashid-alafasy", "slug": "alafasy", "reciter_id": 1, "collection_id": 1},
    {"name": "abdul-rahman-al-sudais", "slug": "sudais", "reciter_id": 12, "collection_id": 1},
    {"name": "abu-bakr-al-shatri", "slug": "shatri", "reciter_id": 4, "collection_id": 1},
    {"name": "saad-al-ghamidi", "slug": "ghamidi", "reciter_id": 6, "collection_id": 1},
    {"name": "maher-al-mueaqly", "slug": "mueaqly", "reciter_id": 13, "collection_id": 1},
    {"name": "saud-al-shuraim", "slug": "shuraim", "reciter_id": 15, "collection_id": 1},
    {"name": "yasser-al-dossari", "slug": "dossari", "reciter_id": 16, "collection_id": 1},
]

async def download_reciter_async(reciter: dict, out_dir: str, max_concurrent: int = 10) -> int:
    """Download all 114 surahs for a reciter via assabile API with concurrency control."""
    os.makedirs(out_dir, exist_ok=True)
    rec_id = reciter["reciter_id"]
    coll_id = reciter.get("collection_id", 1)
    
    # Check already present files
    existing = {int(os.path.basename(f).split(".")[0]) for f in glob.glob(f"{out_dir}/*.mp3")}
    if len(existing) == 114:
        return 114

    async with AsyncSession(timeout=15, impersonate="chrome120") as session:
        lp_url = f"https://www.assabile.com/ajax/loadplayer-{rec_id}-{coll_id}"
        resp = await session.get(lp_url)
        if resp.status_code != 200:
            print(f"  ⚠ Failed to load recitations for {reciter['name']} (status {resp.status_code})")
            return len(existing)
        
        recs = resp.json().get("Recitation", [])
        sem = asyncio.Semaphore(max_concurrent)
        
        async def fetch_surah(r):
            surah_id = r.get("sura_id")
            href = r.get("href", "").lstrip("#")
            if not surah_id or not href:
                return False
            s_num = int(surah_id)
            target_path = os.path.join(out_dir, f"{s_num:03d}.mp3")
            if os.path.exists(target_path) and os.path.getsize(target_path) > 1000:
                return True
            
            async with sem:
                for attempt in range(3):
                    try:
                        r_link = await session.get(f"https://www.assabile.com/ajax/getrcita-link-{href}", timeout=10)
                        if r_link.status_code == 200 and r_link.text.strip().startswith("http"):
                            mp3_url = r_link.text.strip()
                            r_audio = await session.get(mp3_url, timeout=45)
                            if r_audio.status_code == 200 and len(r_audio.content) > 1000:
                                with open(target_path, "wb") as f:
                                    f.write(r_audio.content)
                                return True
                    except Exception:
                        await asyncio.sleep(1.0 * (attempt + 1))
            return False

        tasks = [fetch_surah(r) for r in recs]
        results = await asyncio.gather(*tasks)
        return sum(1 for res in results if res)

print(f"✓ Configured {len(RECITERS)} reciters.")

In [ ]:
# 5. Overlapped Pipelined Execution: Transcode + Fbank + Batched GPU Forced Alignment
import time
from quran_forced_align.batch_cli import run_pipelined_batch

BATCH_SIZE = 8           # Optimal for Colab T4 GPU VRAM
PREFETCH_WORKERS = 4     # CPU threads decoding & extracting Fbank ahead of GPU
PREFETCH_BATCHES = 2     # Double-buffering queue depth

reciter_summaries = []

for r in RECITERS:
    rec_name = r["slug"]
    local_rec_audio = os.path.join(LOCAL_AUDIO, rec_name)
    drive_rec_audio = os.path.join(DRIVE_AUDIO, rec_name)
    drive_rec_json = os.path.join(DRIVE_JSON, rec_name)
    drive_rec_opus = os.path.join(DRIVE_OPUS, rec_name)
    
    os.makedirs(local_rec_audio, exist_ok=True)
    os.makedirs(drive_rec_json, exist_ok=True)
    os.makedirs(drive_rec_opus, exist_ok=True)
    
    # Check if already processed (all 114 JSONs in Drive)
    existing_json = glob.glob(f"{drive_rec_json}/*.json")
    if len(existing_json) == 114:
        print(f"\n[SKIP] {rec_name}: All 114 surahs already aligned in Drive.")
        reciter_summaries.append({
            "reciter": rec_name,
            "status": "already_completed",
            "surahs_succeeded": 114,
            "surahs_failed": 0,
            "audio_hours": 0.0,
            "wall_sec": 0.0,
            "rtf": 0.0,
        })
        continue
    
    print(f"\n{'='*70}\n[START] Reciter: {rec_name} ({r['name']})\n{'='*70}")
    
    # Step 1: Ensure audio downloaded to fast local NVMe
    print(f"[1/3] Downloading/prefetching audio to {local_rec_audio}...")
    num_audio = asyncio.run(download_reciter_async(r, local_rec_audio))
    print(f"      {num_audio}/114 surah audio files ready.")
    
    available = sorted([int(os.path.basename(f).split(".")[0]) for f in glob.glob(f"{local_rec_audio}/*.mp3")])
    if not available:
        print(f"  ⚠ No audio available for {rec_name}, skipping.")
        continue
        
    # Step 2: Run high-throughput pipelined batch execution
    print(f"[2/3] Running Overlapped GPU Alignment & Loudnorm Transcoding (Batch size: {BATCH_SIZE})...")
    t0 = time.monotonic()
    
    summary = run_pipelined_batch(
        surah_list=available,
        audio_dir=local_rec_audio,
        out_dir=drive_rec_json,
        opus_dir=drive_rec_opus,
        transcode_opus=True,
        model_path=local_model_path,
        tokens_path=local_tokens_path,
        device="cuda",
        cuda_batch_size=BATCH_SIZE,
        intra_surah_split=True,
        prefetch_workers=PREFETCH_WORKERS,
        prefetch_batches=PREFETCH_BATCHES,
        verbose=True,
    )
    
    elapsed = time.monotonic() - t0
    audio_hrs = summary["total_audio_sec"] / 3600.0
    rtf = summary["overall_rtf"]
    speedup = 1.0 / max(rtf, 1e-6)
    
    print(f"[3/3] Completed {rec_name}: {summary['succeeded_count']}/{len(available)} surahs in {elapsed/60:.2f}m "
          f"(Audio: {audio_hrs:.2f}h | RTF: {rtf:.4f}x | {speedup:.1f}x realtime)")
    
    reciter_summaries.append({
        "reciter": rec_name,
        "status": "ok" if summary["failed_count"] == 0 else "partial",
        "surahs_succeeded": summary["succeeded_count"],
        "surahs_failed": summary["failed_count"],
        "audio_hours": audio_hrs,
        "wall_sec": elapsed,
        "rtf": rtf,
        "words": summary["total_words"],
        "repeats": summary["total_repeats"],
    })

In [ ]:
# 6. Comprehensive Summary Report
print("\n" + "=" * 88)
print(f"{'Reciter':<25} {'Surahs':<10} {'Audio (h)':<10} {'Wall (m)':<10} {'RTF':<10} {'Speedup':<10} {'Status'}")
print("-" * 88)

tot_audio = 0.0
tot_wall = 0.0
tot_words = 0

for s in reciter_summaries:
    rec = s["reciter"]
    surahs_str = f"{s['surahs_succeeded']}/114"
    audio_h = s.get("audio_hours", 0.0)
    wall_m = s.get("wall_sec", 0.0) / 60.0
    rtf = s.get("rtf", 0.0)
    speed = f"{1.0/max(rtf, 1e-6):.1f}x" if rtf > 0 else "-"
    status = s.get("status", "ok")
    
    tot_audio += audio_h
    tot_wall += s.get("wall_sec", 0.0)
    tot_words += s.get("words", 0)
    
    print(f"{rec:<25} {surahs_str:<10} {audio_h:<10.2f} {wall_m:<10.2f} {rtf:<10.4f} {speed:<10} {status}")

print("=" * 88)
overall_rtf = (tot_wall / max(tot_audio * 3600, 1.0)) if tot_audio > 0 else 0.0
overall_speedup = (1.0 / max(overall_rtf, 1e-6)) if overall_rtf > 0 else 0.0

print(f"TOTALS: {len(reciter_summaries)} reciters | {tot_audio:.2f} hours audio aligned in {tot_wall/60:.2f} mins")
print(f"Overall RTF: {overall_rtf:.4f}x ({overall_speedup:.1f}x realtime speed) | Words aligned: {tot_words:,}")
print(f"Google Drive Outputs: {DRIVE_ROOT}")
print("=" * 88)